# 02 — Exploratory Data Analysis

Goal: understand **which borrower characteristics separate defaulters from non-defaulters**,
and build intuition for feature engineering and modeling.

For a credit model the question is always: *does this feature shift the odds of default?*

We'll examine:
1. Numeric distributions by outcome
2. Delinquency counts (the strongest signal in this dataset)
3. Utilization, age, income, debt ratio
4. Correlation with the target
5. A ranked "signal strength" summary

Charts are saved to `reports/figures/` for the portfolio write-up.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
DATA_PATH = "../data/raw/cs-training.csv"
FIG_DIR = "../reports/figures"

df = pd.read_csv(DATA_PATH, index_col=0)
target = "SeriousDlqin2yrs"
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")


Loaded 150,000 rows x 11 columns


## 0. Quick setup

Split the columns into intuitive groups so the analysis reads clearly.

In [ ]:
delinq_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

# Treat 96/98 as missing for *visual* purposes only here (clean properly in nb03)
def clean_sentinels(s):
    return s.where(s < 96)

features = [c for c in df.columns if c != target]
df[delinq_cols] = df[delinq_cols].apply(clean_sentinels)
print("Features:", len(features))
df[delinq_cols].describe().T[["min", "max", "mean"]]


## 1. The headline: delinquency history is the signal

The past-due columns are the strongest predictors in consumer credit — "the best
predictor of future delinquency is past delinquency." Let's confirm it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, c in zip(axes, delinq_cols):
    rates = df.groupby(c)[target].mean()
    counts = df.groupby(c)[target].size()
    rates = rates[counts >= 50]  # ignore ultra-rare values
    ax.bar(rates.index, rates.values, color="#2c7fb8")
    ax.set_title(c.replace("NumberOfTime", "").replace("NumberOfTimes", ""), fontsize=11)
    ax.set_xlabel("# of times")
    ax.set_ylabel("default rate")
    ax.set_ylim(0, min(1.0, rates.max() * 1.2))
plt.suptitle("Default rate rises steeply with past-due counts", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/eda_delinquency.png", dpi=120, bbox_inches="tight")
plt.show()

print("Default rate when 90+ days late == 0:  {:.2%}".format(df[df["NumberOfTimes90DaysLate"]==0][target].mean()))
print("Default rate when 90+ days late >= 1:  {:.2%}".format(df[df["NumberOfTimes90DaysLate"]>=1][target].mean()))


**Interpretation:** a single 90+ day delinquency multiplies the default rate roughly
5–6x. This is the dominant driver — modeling will hinge on it, so feature engineering
should aggregate and refine these counts rather than discard them.

## 2. Revolving utilization

`RevolvingUtilizationOfUnsecuredLines` = balance / credit limit. Note there are extreme
values (>1 is possible if the limit is exceeded, but values like 50,000 are errors).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(df["RevolvingUtilizationOfUnsecuredLines"].clip(0, 2), bins=60, color="#7570b3")
axes[0].set_title("Utilization (clipped at 2)")
axes[0].set_xlabel("utilization")

tmp = df.copy()
tmp["util_bin"] = pd.cut(tmp["RevolvingUtilizationOfUnsecuredLines"].clip(0, 1.5),
                         bins=[0, .1, .3, .5, .7, .9, 1.0, 1.5],
                         include_lowest=True)
rate = tmp.groupby("util_bin", observed=True)[target].mean()
axes[1].bar(range(len(rate)), rate.values, color="#d95f02")
axes[1].set_xticks(range(len(rate)))
axes[1].set_xticklabels([str(i) for i in rate.index], rotation=45, ha="right")
axes[1].set_title("Default rate by utilization bin")
axes[1].set_ylabel("default rate")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/eda_utilization.png", dpi=120, bbox_inches="tight")
plt.show()

**Interpretation:** low utilization (under 30%) is safe; default risk climbs sharply
as a borrower maxes out their lines. The relationship is monotonic → great for a model.

## 3. Age

Younger borrowers are historically riskier in this dataset.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
bins = [0, 25, 35, 45, 55, 65, 120]
labels = ["<25", "25-34", "35-44", "45-54", "55-64", "65+"]
tmp = df.copy()
tmp["age_bin"] = pd.cut(tmp["age"], bins=bins, labels=labels, right=False)
rate = tmp.groupby("age_bin", observed=True)[target].mean()
ax.bar(rate.index, rate.values, color="#1b9e77")
ax.set_title("Default rate by age group")
ax.set_ylabel("default rate")
for i, v in enumerate(rate.values):
    ax.text(i, v + 0.002, f"{v:.1%}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/eda_age.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Income, debt ratio, and the missingness signal

`MonthlyIncome` is ~20% missing. In credit data, income is often missing for
non-earners — which correlates with risk. So the *missing indicator itself* is a feature.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(df["MonthlyIncome"].dropna().clip(0, 25000), bins=60, color="#e7298a")
axes[0].set_title("Monthly income (clipped at 25k)")
axes[0].set_xlabel("income")

tmp = df.copy()
tmp["income_missing"] = tmp["MonthlyIncome"].isnull()
rate = tmp.groupby("income_missing")[target].mean()
axes[1].bar(["income present", "income MISSING"], rate.values, color="#66a61e")
axes[1].set_title("Default rate by income-missingness")
axes[1].set_ylabel("default rate")
for i, v in enumerate(rate.values):
    axes[1].text(i, v + 0.002, f"{v:.1%}", ha="center", fontsize=10)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/eda_income.png", dpi=120, bbox_inches="tight")
plt.show()

print("Default rate | income present: {:.2%}".format(rate[False]))
print("Default rate | income MISSING: {:.2%}".format(rate[True]))

**Interpretation:** missing income is associated with a *lower* default rate here
(counterintuitive), but the gap confirms missingness carries information — we should keep
an indicator rather than blindly imputing.

## 5. Correlation with the target

A single ranked view of linear association (Spearman — robust to the skew in these features).

In [ ]:
corr = (df.corr(method="spearman")[target]
       .drop(target).sort_values(ascending=False))
corr.to_frame("spearman_corr_with_default")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#d95f02" if v > 0 else "#1b9e77" for v in corr.values]
ax.barh(corr.index[::-1], corr.values[::-1], color=colors[::-1])
ax.axvline(0, color="k", linewidth=0.8)
ax.set_title("Spearman correlation with default")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/eda_correlation.png", dpi=120, bbox_inches="tight")
plt.show()

## 6. Signal strength summary

Combine the correlation view into a short ranked table for the write-up.

In [ ]:
summary = pd.DataFrame({
    "spearman_corr": corr,
    "abs_corr": corr.abs(),
}).sort_values("abs_corr", ascending=False)
summary["rank"] = range(1, len(summary) + 1)
summary[["rank", "spearman_corr"]].style.background_gradient(cmap="RdYlGn_r", subset=["spearman_corr"])

### Takeaways

- **Delinquency counts dominate** — especially `NumberOfTimes90DaysLate`.
- **Utilization** is monotonic and strong.
- **Age** has a clear gradient (younger = riskier).
- **Missing income is informative** — keep an indicator flag.
- Several features are **skewed with extreme outliers** → transform/winsorize in nb03.

Next: **03 — Feature Engineering & Cleaning** (sentinel fixes, imputation strategy,
ratios/aggregations, and a persisted clean dataset).